# SmartBag — MLP / TinyML

Mesmo CSV, mesmas seis features, mesmas classes e **a mesma divisao por rodadas** da
Random Forest — so assim as metricas dos dois modelos sao comparaveis.

Rede pequena: 6 -> 8 -> 1. Normalizacao fora da rede, repetida no ESP32.

| Arquivo | Vai para | Serve para |
|---|---|---|
| `modelo_smartbag.h` | app da MLP embarcada `device/src/` | os bytes do `.tflite` |
| `AIoTSmartBagScaler.hpp` | app da MLP embarcada `device/src/` | normalizacao, identica a do app da floresta embarcada |

## 1. Pacotes

In [ ]:
# numpy/pandas/scikit-learn nas MESMAS versoes do notebook da RF: e o que garante
# que o StandardScaler daqui produza media e escala identicas as de la, e que os dois
# apps embarcados possam usar o mesmo header.
%pip install -q matplotlib "numpy==2.1.3" "pandas==2.2.3" "scikit-learn==1.6.1" "tensorflow==2.20.0"

In [ ]:
import json, hashlib, zipfile
from pathlib import Path
from importlib.metadata import version
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay
import tensorflow as tf
from sklearn.preprocessing import StandardScaler
tf.keras.utils.set_random_seed(42)


## 2. Abrir e conferir o CSV

Mesmo mapa da RF: `ENTREGA_OK = 0`, `REVISAR_ENTREGA = 1`. Aqui a saida e uma
probabilidade, e 1 e a classe positiva — a rede responde "qual a chance de precisar
revisar".

In [ ]:
from google.colab import files
arquivos = files.upload()
ARQUIVO = next(iter(arquivos))
df = pd.read_csv(ARQUIVO).sort_values(["rodada", "situacao", "timestamp"]).reset_index(drop=True)

FEATURES = ["temperatura", "umidade", "delta_distancia", "luz", "mov_max", "incl_max"]
CLASSES = ["ENTREGA_OK", "REVISAR_ENTREGA"]   # indice = o codigo: 0 e 1
MAPA = {nome: i for i, nome in enumerate(CLASSES)}

X = df[FEATURES].astype(np.float32)
y = df["target"].map(MAPA)
assert y.notna().all(), "Ha target fora de CLASSES. Confira o CSV; nao converta em 0 silenciosamente."
assert df["device"].nunique() == 1, "Use uma execucao de uma equipe."
print("Mapa de classes:", MAPA)
display(pd.crosstab(df["rodada"], df["target"]))

## 3. Separar por rodada
A última rodada fica no teste, como no app de coleta do motor. RF e MLP usam o mesmo CSV e a mesma divisão.


In [ ]:
rodada_teste = df["rodada"].max()
indices_treino = df.index[df["rodada"] != rodada_teste]
indices_teste = df.index[df["rodada"] == rodada_teste]
X_treino, X_teste = X.loc[indices_treino], X.loc[indices_teste]
y_treino, y_teste = y.loc[indices_treino], y.loc[indices_teste]
rodadas_treino = sorted(df.loc[indices_treino, "rodada"].unique().tolist())
rodadas_teste = [int(rodada_teste)]
assert set(y_treino) == set(y_teste) == {0, 1}, "Colete pelo menos duas rodadas completas com as duas classes."
print("Treino:", rodadas_treino, "| Teste:", rodadas_teste)

## 4. Normalizar e treinar

Scaler ajustado **so no treino** — ajustar no CSV inteiro vazaria a media da rodada de
teste para dentro do modelo.

Aqui a normalizacao nao e simetria com o app da floresta embarcada: e **necessidade**. A rede soma entradas
multiplicadas por pesos, e `luz` vai a 4095 enquanto `mov_max` fica abaixo de 20. Sem
normalizar, a luz domina o gradiente e as outras cinco features quase nao treinam.

Sao 50 epocas fixas. Com duas rodadas nao ha conjunto de validacao independente, entao o
teste nao participa do fit nem de early stopping — olhar o teste para decidir quando
parar e uma forma de treinar nele.

In [ ]:
scaler = StandardScaler().fit(X_treino)
X_treino_norm = scaler.transform(X_treino).astype(np.float32)
X_teste_norm = scaler.transform(X_teste).astype(np.float32)
modelo = tf.keras.Sequential([
    tf.keras.Input(shape=(6,)),
    tf.keras.layers.Dense(8, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid"),
])
modelo.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
modelo.fit(X_treino_norm, y_treino.to_numpy(), epochs=50, batch_size=32, verbose=0)
modelo.summary()
probabilidades = modelo.predict(X_teste_norm, verbose=0).ravel()
predicoes = (probabilidades >= 0.5).astype(int)


## 5. Avaliar

In [ ]:
print("Acuracia:", accuracy_score(y_teste, predicoes))
print(classification_report(y_teste, predicoes, labels=[0, 1], target_names=CLASSES, zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_teste, predicoes, labels=[0, 1], display_labels=CLASSES, cmap="Blues")
plt.show()
display(pd.crosstab(df.loc[indices_teste, "situacao"],
                    pd.Series(predicoes, index=y_teste.index, name="predicao")))

## 6. Converter e conferir TFLite
Float32, sem quantização. A entrada já está normalizada.

In [ ]:
conversor = tf.lite.TFLiteConverter.from_keras_model(modelo)
tflite = conversor.convert()
Path("modelo_smartbag.tflite").write_bytes(tflite)
interpreter = tf.lite.Interpreter(model_content=tflite)
interpreter.allocate_tensors()
entrada = interpreter.get_input_details()[0]
saida = interpreter.get_output_details()[0]
assert list(entrada["shape"]) == [1, 6] and list(saida["shape"]) == [1, 1]
assert entrada["dtype"] == np.float32 and saida["dtype"] == np.float32
prob_tflite = []
for x in X_teste_norm:
    interpreter.set_tensor(entrada["index"], x.reshape(1, 6))
    interpreter.invoke()
    prob_tflite.append(float(interpreter.get_tensor(saida["index"])[0, 0]))
prob_tflite = np.array(prob_tflite)
np.testing.assert_allclose(prob_tflite, probabilidades, atol=1e-5, rtol=1e-5)
assert np.array_equal(prob_tflite >= 0.5, predicoes), "Divergência de classe perto de 0,5: inspecione antes de exportar."
print("Keras e TFLite conferidos. Bytes:", len(tflite))


## 7. Exportar modelo e scaler

No ESP32: ler os seis valores originais, `Scaler::standardize()` **uma vez**, alimentar as seis
entradas e decidir `REVISAR_ENTREGA` quando a saida for >= 0,5.

O header do scaler sai com o mesmo nome e a mesma funcao do app da floresta embarcada — e com os mesmos
numeros, porque o split e o `StandardScaler` sao os mesmos. Confira os vetores impressos
contra os do notebook da RF.

In [ ]:
bytes_cpp = ", ".join(f"0x{b:02x}" for b in tflite)
Path("modelo_smartbag.h").write_text(
    "#pragma once\n#include <stdint.h>\nalignas(16) const unsigned char modelo_smartbag_tflite[] = {"
    + bytes_cpp + "};\n",
    encoding="utf-8")

# Mesmo formato do app de ocupação com micromlgen e do app da floresta embarcada: Scaler::standardize(entrada, saida).
media = scaler.mean_.astype(np.float32)
escala = scaler.scale_.astype(np.float32)
cabecalho = '''#pragma once

// StandardScaler ajustado no treino do este app (notebook da MLP).
// Ordem: temperatura, umidade, delta_distancia, luz, mov_max, incl_max.
namespace Scaler {
    const float media[6] = {__MEDIA__};
    const float escala[6] = {__ESCALA__};

    void standardize(const float entrada[6], float saida[6]) {
        for (int i = 0; i < 6; i++) saida[i] = (entrada[i] - media[i]) / escala[i];
    }
}
'''.replace("__MEDIA__", ", ".join(f"{v:.9e}f" for v in media)) \
   .replace("__ESCALA__", ", ".join(f"{v:.9e}f" for v in escala))
Path("AIoTSmartBagScaler.hpp").write_text(cabecalho, encoding="utf-8")

# A conta que o ESP32 vai fazer, conferida contra a que o scikit-learn fez.
normalizado_esp = (X_teste.to_numpy(np.float32) - media) / escala
np.testing.assert_allclose(normalizado_esp, X_teste_norm, atol=1e-5, rtol=1e-5)

PACOTES = ["tensorflow", "scikit-learn", "numpy", "pandas"]
metadados = {
    "features": FEATURES,
    "unidades": ["C", "%", "cm", "RAW (0..4095)", "m/s2", "graus"],
    "classes": CLASSES,
    "mapa_classes": MAPA,
    "saida": "sigmoide; >= limiar e REVISAR_ENTREGA",
    "limiar": 0.5,
    "media": media.tolist(),
    "escala": escala.tolist(),
    "rodadas_treino": rodadas_treino,
    "rodadas_teste": rodadas_teste,
    "csv_sha256": hashlib.sha256(Path(ARQUIVO).read_bytes()).hexdigest(),
    "versoes": {p: version(p) for p in PACOTES},
}
Path("metadados_mlp.json").write_text(json.dumps(metadados, indent=2), encoding="utf-8")

conferencia = X_teste.copy()
conferencia["probabilidade"] = prob_tflite
conferencia["classe"] = predicoes
conferencia.to_csv("conferencia_mlp.csv", index=False)

print("media: ", np.round(media, 4))
print("escala:", np.round(escala, 4))
print("Confira: devem ser IDENTICOS aos do notebook da Random Forest.")

## 8. Baixar

O app da MLP embarcada usa `modelo_smartbag.h` e `AIoTSmartBagScaler.hpp`.

O teste do TFLite feito aqui roda no interpretador do PC. Ele **nao** substitui o ensaio
no ESP32: la o interpretador e o MicroTFLite, com arena fixa, e a inicializacao pode
falhar por memoria sem que nada aqui acuse.

In [ ]:
with zipfile.ZipFile("smartbag_mlp.zip", "w") as pacote:
    for nome in ["modelo_smartbag.tflite", "modelo_smartbag.h", "AIoTSmartBagScaler.hpp", "metadados_mlp.json", "conferencia_mlp.csv"]:
        pacote.write(nome)
files.download("smartbag_mlp.zip")
